In [1]:
import numpy as np
import torch
import torch.nn as nn 

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix

In [2]:
SEED = 42

torch.set_num_threads(2)
torch.set_num_interop_threads(1)

np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA kullanılamıyor. Eğitim CPU üzerinde başlatılmayacak."
    )

torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0")
torch.cuda.set_device(0)

print("Selected device:", device)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.12.1+cu130
PyTorch CUDA build: 13.0
CUDA available: True
CUDA device count: 1
Selected device: cuda:0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
train_images_path = "archive/train-images.idx3-ubyte"
train_labels_path = "archive/train-labels.idx1-ubyte"

test_images_path = "archive/t10k-images.idx3-ubyte"
test_labels_path = "archive/t10k-labels.idx1-ubyte"

In [4]:
def read_images(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(16), dtype=">u4")
        magic_number, image_count, row_count, column_count = header

        pixel_data = file.read()
        pixels = np.frombuffer(pixel_data, dtype=np.uint8)

        images = pixels.reshape(image_count, row_count, column_count)
    
    return images, magic_number, image_count, row_count, column_count

In [5]:
def read_labels(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(8), dtype=">u4")
        magic_number, label_count = header

        label_data = file.read()
        labels = np.frombuffer(label_data, dtype=np.uint8)

    return labels, magic_number, label_count

In [6]:
train_images, train_image_magic_number, train_image_count, train_image_row_count, train_image_column_count = read_images(train_images_path)
train_labels, train_label_magic_number, train_label_count = read_labels(train_labels_path)

test_images, test_image_magic_number, test_image_count, test_image_row_count, test_image_column_count = read_images(test_images_path)
test_labels, test_label_magic_number, test_label_count = read_labels(test_labels_path)

print("Train images:", train_images.shape)
print("Train labels:", train_labels.shape)
print("Test images:", test_images.shape)
print("Test labels:", test_labels.shape)

print("Train image magic number:", train_image_magic_number)
print("Train label magic number:", train_label_magic_number)
print("Test image magic number:", test_image_magic_number)
print("Test label magic number:", test_label_magic_number)

Train images: (60000, 28, 28)
Train labels: (60000,)
Test images: (10000, 28, 28)
Test labels: (10000,)
Train image magic number: 2051
Train label magic number: 2049
Test image magic number: 2051
Test label magic number: 2049


In [7]:
train_vectors = train_images.reshape(train_images.shape[0], train_images.shape[1] * train_images.shape[2])
test_vectors = test_images.reshape(test_images.shape[0], test_images.shape[1] * test_images.shape[2])

normalized_train_vectors = train_vectors.astype(np.float32) / 255.0
normalized_test_vectors = test_vectors.astype(np.float32) / 255.0

In [8]:
validation_per_class = 1000

random_generator = np.random.default_rng(SEED)

validation_indices_list = []
training_indices_list = []

for class_label in range(10):
    class_indices = np.where(train_labels == class_label)[0]
    shuffled_class_indices = random_generator.permutation(class_indices)

    validation_indices_list.append(shuffled_class_indices[:validation_per_class])
    training_indices_list.append(shuffled_class_indices[validation_per_class:])

validation_indices = np.concatenate(validation_indices_list)
training_indices = np.concatenate(training_indices_list)

validation_indices = random_generator.permutation(validation_indices)
training_indices = random_generator.permutation(training_indices)

model_train_vectors = normalized_train_vectors[training_indices]
model_train_labels = train_labels[training_indices]

validation_vectors = normalized_train_vectors[validation_indices]
validation_labels = train_labels[validation_indices]

In [9]:
model_train_tensor = torch.from_numpy(model_train_vectors)
model_train_label_tensor = torch.from_numpy(model_train_labels.astype(np.int64))

validation_tensor = torch.from_numpy(validation_vectors)
validation_label_tensor = torch.from_numpy(validation_labels.astype(np.int64))

test_tensor = torch.from_numpy(normalized_test_vectors)
test_label_tensor = torch.from_numpy(test_labels.astype(np.int64))

In [10]:
model_train_dataset = TensorDataset(model_train_tensor, model_train_label_tensor)
validation_dataset = TensorDataset(validation_tensor, validation_label_tensor)
test_dataset = TensorDataset(test_tensor, test_label_tensor)

In [11]:
def create_data_loaders(training_batch_size, evaluation_batch_size=1024, seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)

    pin_memory = device.type == "cuda"

    training_loader = DataLoader(model_train_dataset, batch_size=training_batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory, generator=generator)
    validation_loader = DataLoader(validation_dataset, batch_size=evaluation_batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=evaluation_batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)

    return training_loader, validation_loader, test_loader

In [12]:
BASELINE_BATCH_SIZE = 128

baseline_training_loader, baseline_validation_loader, test_loader = (create_data_loaders(training_batch_size=BASELINE_BATCH_SIZE))

In [13]:
def create_activation(activation_name):

    if activation_name == "relu":
        return nn.ReLU()
    if activation_name == "leaky_relu":
        return nn.LeakyReLU(negative_slope=0.01)
    if activation_name == "elu":
        return nn.ELU()
    if activation_name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Unsupported activation function: {activation_name}")

In [14]:
class MLP(nn.Module):

    def __init__(self, input_size, hidden_sizes, output_size, activation_name, use_batch_normalization, dropout_rate):
        super().__init__()

        layers = []
        current_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(current_size, hidden_size))

            if use_batch_normalization:
                layers.append(nn.BatchNorm1d(hidden_size))

            layers.append(create_activation(activation_name))

            if dropout_rate > 0.0:
                layers.append(nn.Dropout(dropout_rate))

            current_size = hidden_size

        layers.append(nn.Linear(current_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [15]:
baseline_config = {
    "input_size": 784,
    "hidden_sizes": [512, 256, 128],
    "output_size": 10,
    "activation_name": "relu",
    "use_batch_normalization": True,
    "dropout_rate": 0.20
}

baseline_model = MLP(**baseline_config).to(device)

In [16]:
class_counts = np.bincount(model_train_labels, minlength=10)

class_weights = (len(model_train_labels) / (10 * class_counts))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Class counts: ", class_counts)
print("Class weights: ", class_weights)

Class counts:  [4923 5742 4958 5131 4842 4421 4918 5265 4851 4949]
Class weights:  [1.01564087 0.87077673 1.00847116 0.97446891 1.03263114 1.13096584
 1.01667344 0.94966762 1.03071532 1.01030511]


In [17]:
BASELINE_LEARNING_RATE = 0.001

baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=BASELINE_LEARNING_RATE)

In [18]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), dtype=torch.long, device=device)
    total_samples = 0

    for feature_batch, label_batch in data_loader:
        feature_batch = feature_batch.to(device, non_blocking=True)
        label_batch = label_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        output_batch = model(feature_batch)
        loss = criterion(output_batch, label_batch)

        loss.backward()
        optimizer.step()

        batch_size = label_batch.size(0)

        total_loss += loss.detach() * batch_size

        total_correct += (output_batch.argmax(dim=1) == label_batch).sum()

        total_samples += batch_size
    
    average_loss = (total_loss / total_samples).item()
    accuracy = (total_correct.float() / total_samples).item()

    return average_loss, accuracy


In [19]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), dtype=torch.long, device=device)
    total_samples = 0

    with torch.inference_mode():
        for feature_batch, label_batch in data_loader:
            feature_batch = feature_batch.to(device, non_blocking=True)
            label_batch = label_batch.to(device, non_blocking=True)
            output_batch = model(feature_batch)
            loss = criterion(output_batch, label_batch)

            batch_size = label_batch.size(0)
            total_loss += loss * batch_size
            total_correct += (output_batch.argmax(dim=1) == label_batch).sum()
            total_samples += batch_size

    average_loss = (total_loss / total_samples).item()
    accuracy = (total_correct.float() / total_samples).item()

    return average_loss, accuracy

In [20]:
def train_with_early_stopping(model, training_loader, validation_loader, criterion, optimizer, device, maximum_epochs, patience, checkpoint_path, verbose=True):
    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": []
    }

    for epoch in range(1, maximum_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(
            model=model,
            data_loader=training_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )

        validation_loss, validation_accuracy = evaluate_model(
            model=model,
            data_loader=validation_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), checkpoint_path)

        else:
            epochs_without_improvement += 1

        if verbose:
            print(
                f"Epoch {epoch:02d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Train acc: {train_accuracy:.4f} | "
                f"Validation loss: {validation_loss:.4f} | "
                f"Validation acc: {validation_accuracy:.4f}"
            )

        if epochs_without_improvement >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}. "
                    f"Best epoch: {best_epoch}"
                )
            break

    best_model_state = torch.load(checkpoint_path, weights_only=True, map_location=device)
    model.load_state_dict(best_model_state)
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch, 
        "best_validation_loss": best_validation_loss
    }

In [21]:
baseline_training_result = train_with_early_stopping(
    model=baseline_model,
    training_loader=baseline_training_loader,
    validation_loader=baseline_validation_loader,
    criterion=criterion,
    optimizer=baseline_optimizer,
    device=device,
    maximum_epochs=60,
    patience=8,
    checkpoint_path="baseline_best_model.pt",
    verbose=True
)

Epoch 01 | Train loss: 0.2852 | Train acc: 0.9246 | Validation loss: 0.1172 | Validation acc: 0.9662
Epoch 02 | Train loss: 0.1179 | Train acc: 0.9650 | Validation loss: 0.0859 | Validation acc: 0.9749
Epoch 03 | Train loss: 0.0888 | Train acc: 0.9725 | Validation loss: 0.0832 | Validation acc: 0.9749
Epoch 04 | Train loss: 0.0711 | Train acc: 0.9774 | Validation loss: 0.0756 | Validation acc: 0.9780
Epoch 05 | Train loss: 0.0607 | Train acc: 0.9809 | Validation loss: 0.0705 | Validation acc: 0.9801
Epoch 06 | Train loss: 0.0504 | Train acc: 0.9843 | Validation loss: 0.0768 | Validation acc: 0.9782
Epoch 07 | Train loss: 0.0464 | Train acc: 0.9847 | Validation loss: 0.0721 | Validation acc: 0.9795
Epoch 08 | Train loss: 0.0413 | Train acc: 0.9870 | Validation loss: 0.0693 | Validation acc: 0.9802
Epoch 09 | Train loss: 0.0384 | Train acc: 0.9873 | Validation loss: 0.0733 | Validation acc: 0.9798
Epoch 10 | Train loss: 0.0348 | Train acc: 0.9887 | Validation loss: 0.0700 | Validation ac

In [22]:
print("Selected device: ", device)
print("Baseline model device: ", next(baseline_model.parameters()).device)
print("GPU: ", torch.cuda.get_device_name(0))
print("CUDA memory allocated: ", round(torch.cuda.memory_allocated(0) / (1024 ** 2), 2), "MB")

Selected device:  cuda:0
Baseline model device:  cuda:0
GPU:  NVIDIA GeForce RTX 4060 Laptop GPU
CUDA memory allocated:  25.63 MB


In [23]:
best_baseline_model = baseline_training_result["model"]

baseline_validation_loss, baseline_validation_accuracy = (
    evaluate_model(
        model=best_baseline_model,
        data_loader=baseline_validation_loader,
        criterion=criterion,
        device=device
    )
)

print("Baseline best epoch:", baseline_training_result["best_epoch"])
print("Recorded best validation loss:", baseline_training_result["best_validation_loss"])
print("Reloaded baseline validation loss:", baseline_validation_loss)
print("Reloaded baseline validation accuracy:", baseline_validation_accuracy)
print("Baseline validation accuracy percentage:", f"{baseline_validation_accuracy * 100:.2f}%")

Baseline best epoch: 14
Recorded best validation loss: 0.06188831478357315
Reloaded baseline validation loss: 0.06188831478357315
Reloaded baseline validation accuracy: 0.9836999773979187
Baseline validation accuracy percentage: 98.37%


In [24]:
def set_experiment_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_experiment(config, experiment_name, maximum_epochs=40, seed=SEED, verbose=False):
    set_experiment_seed(seed)

    training_loader, validation_loader, _ = create_data_loaders(training_batch_size=config["batch_size"], seed=seed)
    model = MLP(
        input_size=784,
        hidden_sizes=config["hidden_sizes"],
        output_size=10,
        activation_name=config["activation_name"],
        use_batch_normalization=config["use_batch_normalization"],
        dropout_rate=config["dropout_rate"]
    ).to(device)

    if device.type != "cuda":
        raise RuntimeError(
            "Model CPU üzerinde çalıştırılmaya çalışıldı."
        )

    assert next(model.parameters()).is_cuda, (
        "Model parametreleri GPU üzerinde değil."
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    checkpoint_path = f"{experiment_name}_best_model.pt"

    training_result = train_with_early_stopping(
        model=model,
        training_loader=training_loader,
        validation_loader=validation_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        maximum_epochs=maximum_epochs,
        patience=config["patience"],
        checkpoint_path=checkpoint_path,
        verbose=verbose
    )

    best_model = training_result["model"]

    validation_loss, validation_accuracy =evaluate_model(
        model=best_model,
        data_loader=validation_loader,
        criterion=criterion,
        device=device
    )

    return {
        "experiment_name": experiment_name,
        "config": config.copy(),
        "model": best_model,
        "history": training_result["history"],
        "best_epoch": training_result["best_epoch"],
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "checkpoint_path": checkpoint_path
    }

In [25]:
def round_to_multiple(value, multiple=16):
    return max(multiple, int(round(value / multiple) * multiple))

In [26]:
ACTIVATION_NAMES = ["relu" , "leaky_relu" , "elu" , "tanh"]

def generate_coarse_config(random_generator):
    hidden_layer_count = int(random_generator.integers(1,7))
    first_hidden_size = 2 ** random_generator.uniform(np.log2(64), np.log2(1024))

    hidden_sizes = [round_to_multiple(first_hidden_size)]

    for _ in range(1,  hidden_layer_count):
        shrink_ratio = random_generator.uniform(0.45, 0.90)

        next_hidden_size = (hidden_sizes[-1] * shrink_ratio)

        hidden_sizes.append(max(32, round_to_multiple(next_hidden_size)))

    use_dropout = bool(random_generator.integers(0, 2))

    if use_dropout:
        dropout_rate = float(random_generator.uniform(0.05, 0.50))
    else:
        dropout_rate = 0.0
        
    batch_size = round_to_multiple(random_generator.uniform(32, 512), multiple=8)
    learning_rate = float(10 ** random_generator.uniform(-5, -2))
    patience = int(random_generator.integers(3, 11))
    activation_name = str(random_generator.choice(ACTIVATION_NAMES))
    use_batch_normalization = bool(random_generator.integers(0,2))

    return {
        "hidden_sizes": hidden_sizes,
        "activation_name": activation_name,
        "use_batch_normalization": use_batch_normalization,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "patience": patience
    }

In [27]:
coarse_random_generator = np.random.default_rng(SEED)

example_coarse_configs = [generate_coarse_config(coarse_random_generator) for _ in range(5)]

for config_index, config in enumerate(example_coarse_configs, start=1):
    print(f"Configuration {config_index}")
    print(config)
    print()

Configuration 1
{'hidden_sizes': [224], 'activation_name': 'tanh', 'use_batch_normalization': True, 'dropout_rate': 0.4363690639601221, 'batch_size': 368, 'learning_rate': 1.9166024706981723e-05, 'patience': 7}

Configuration 2
{'hidden_sizes': [560, 288, 192, 112, 96], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.4202427259718735, 'batch_size': 248, 'learning_rate': 4.8052519691574605e-05, 'patience': 8}

Configuration 3
{'hidden_sizes': [640, 464, 368, 224, 192, 160], 'activation_name': 'leaky_relu', 'use_batch_normalization': False, 'dropout_rate': 0.0, 'batch_size': 408, 'learning_rate': 3.8363314625091895e-05, 'patience': 5}

Configuration 4
{'hidden_sizes': [96], 'activation_name': 'tanh', 'use_batch_normalization': False, 'dropout_rate': 0.38514297015851773, 'batch_size': 496, 'learning_rate': 9.494586914206585e-05, 'patience': 8}

Configuration 5
{'hidden_sizes': [112], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 

In [28]:
test_search_config = example_coarse_configs[0]
print(test_search_config)

{'hidden_sizes': [224], 'activation_name': 'tanh', 'use_batch_normalization': True, 'dropout_rate': 0.4363690639601221, 'batch_size': 368, 'learning_rate': 1.9166024706981723e-05, 'patience': 7}


In [29]:
test_experiment_result = run_experiment(
    config=test_search_config,
    experiment_name="search_function_test",
    maximum_epochs=5,
    seed=SEED,
    verbose=True
)

print("Best epoch:", test_experiment_result["best_epoch"])
print("Validation loss:", test_experiment_result["validation_loss"])
print("Validation accuracy:", test_experiment_result["validation_accuracy"])
print("Checkpoint:", test_experiment_result["checkpoint_path"])

Epoch 01 | Train loss: 1.8573 | Train acc: 0.4270 | Validation loss: 1.4195 | Validation acc: 0.7011
Epoch 02 | Train loss: 1.2110 | Train acc: 0.7239 | Validation loss: 1.0185 | Validation acc: 0.7956
Epoch 03 | Train loss: 0.9261 | Train acc: 0.7997 | Validation loss: 0.8164 | Validation acc: 0.8335
Epoch 04 | Train loss: 0.7704 | Train acc: 0.8328 | Validation loss: 0.6979 | Validation acc: 0.8536
Epoch 05 | Train loss: 0.6712 | Train acc: 0.8518 | Validation loss: 0.6197 | Validation acc: 0.8663
Best epoch: 5
Validation loss: 0.6196767687797546
Validation accuracy: 0.8662999868392944
Checkpoint: search_function_test_best_model.pt


In [30]:
BROAD_COARSE_TRIAL_COUNT = 40
BROAD_COARSE_MAXIMUM_EPOCHS = 60

broad_coarse_random_generator = np.random.default_rng(SEED + 100)
broad_coarse_results = []

for trial_index in range(1, BROAD_COARSE_TRIAL_COUNT + 1):
    config = generate_coarse_config(broad_coarse_random_generator)

    print(f"\nBroad coarse trial " f"{trial_index} / {BROAD_COARSE_TRIAL_COUNT}")
    print(config)

    experiment_result = run_experiment(
        config=config,
        experiment_name="broad_coarse_temporary",
        maximum_epochs=BROAD_COARSE_MAXIMUM_EPOCHS,
        seed=SEED,
        verbose=False
    )

    broad_coarse_results.append({
        "trial": trial_index,
        "config": config.copy(),
        "best_epoch": experiment_result["best_epoch"],
        "validation_loss": experiment_result["validation_loss"],
        "validation_accuracy": experiment_result["validation_accuracy"]
    })

    torch.save(broad_coarse_results, "broad_coarse_results.pt")

    print(
        f"Best epoch: {experiment_result['best_epoch']} | "
        f"Validation loss: "
        f"{experiment_result['validation_loss']:.4f} | "
        f"Validation accuracy: "
        f"{experiment_result['validation_accuracy']:.4f}"
    )

    del experiment_result

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    

print(
    "\nCompleted broad coarse trials:",
    len(broad_coarse_results)
)


Broad coarse trial 1 / 40
{'hidden_sizes': [80], 'activation_name': 'tanh', 'use_batch_normalization': False, 'dropout_rate': 0.0, 'batch_size': 344, 'learning_rate': 0.007622447681242248, 'patience': 4}
Best epoch: 8 | Validation loss: 0.1115 | Validation accuracy: 0.9671

Broad coarse trial 2 / 40
{'hidden_sizes': [128, 80, 48, 32, 32, 32], 'activation_name': 'leaky_relu', 'use_batch_normalization': False, 'dropout_rate': 0.0, 'batch_size': 328, 'learning_rate': 0.0016783311558879782, 'patience': 4}
Best epoch: 9 | Validation loss: 0.1195 | Validation accuracy: 0.9684

Broad coarse trial 3 / 40
{'hidden_sizes': [80, 48, 32, 32], 'activation_name': 'tanh', 'use_batch_normalization': False, 'dropout_rate': 0.41996032271342293, 'batch_size': 56, 'learning_rate': 0.00349881536292535, 'patience': 10}
Best epoch: 45 | Validation loss: 0.2600 | Validation accuracy: 0.9404

Broad coarse trial 4 / 40
{'hidden_sizes': [448, 384, 320, 224, 176], 'activation_name': 'leaky_relu', 'use_batch_norm

In [31]:
broad_coarse_results = torch.load(
    "broad_coarse_results.pt",
    weights_only=False
)

broad_coarse_results = sorted(
    broad_coarse_results,
    key=lambda result: (
        result["validation_loss"],
        -result["validation_accuracy"]
    )
)

print("Top 10 broad coarse configurations:\n")

for rank, result in enumerate(broad_coarse_results[:10], start=1):
    print(f"Rank {rank}")
    print("Trial:", result["trial"])
    print("Configuration:", result["config"])
    print("Best epoch:", result["best_epoch"])
    print(
        "Validation loss:",
        round(result["validation_loss"], 6)
    )
    print(
        "Validation accuracy:",
        round(result["validation_accuracy"], 6)
    )
    print()

Top 10 broad coarse configurations:

Rank 1
Trial: 26
Configuration: {'hidden_sizes': [864, 624, 496, 304], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.20185521672972673, 'batch_size': 40, 'learning_rate': 0.0010606140772019422, 'patience': 4}
Best epoch: 12
Validation loss: 0.058247
Validation accuracy: 0.9851

Rank 2
Trial: 33
Configuration: {'hidden_sizes': [592, 336, 288, 144], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.153558734674836, 'batch_size': 248, 'learning_rate': 0.0032870533959588758, 'patience': 9}
Best epoch: 14
Validation loss: 0.062773
Validation accuracy: 0.9836

Rank 3
Trial: 38
Configuration: {'hidden_sizes': [656, 464, 304], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.0, 'batch_size': 376, 'learning_rate': 0.0006802987533807844, 'patience': 7}
Best epoch: 10
Validation loss: 0.066877
Validation accuracy: 0.9831

Rank 4
Trial: 20
Configuration: {'hidden_sizes': [4

In [32]:
def generate_refined_config(center_config, random_generator):
    center_hidden_sizes = center_config["hidden_sizes"]
    center_layer_count = len(center_hidden_sizes)

    min_layer_count = max(1, center_layer_count - 1)
    max_layer_count = min(7, center_layer_count + 1)

    hidden_layer_count = int(random_generator.integers(min_layer_count, max_layer_count + 1))
    hidden_sizes = []

    for layer_index in range(hidden_layer_count):
        if layer_index < center_layer_count:
            center_size = center_hidden_sizes[layer_index]
        else:
            center_size = max(32, center_hidden_sizes[-1] * 0.70)

        min_size = max(32, center_size * 0.80)
        max_size = min(1024, center_size * 1.20)

        hidden_size = round_to_multiple(random_generator.uniform(min_size, max_size), multiple=16)

        if hidden_sizes:
            hidden_size = min(hidden_size, hidden_sizes[-1])

        hidden_sizes.append(max(32, hidden_size))

    center_learning_rate = center_config["learning_rate"]
    min_learning_rate = max(1e-5, center_learning_rate / 2)
    max_learning_rate = min(1e-2, center_learning_rate * 2)
    learning_rate = float(10 ** random_generator.uniform(np.log10(min_learning_rate), np.log10(max_learning_rate)))

    center_batch_size = center_config["batch_size"]
    min_batch_size = max(32, center_batch_size - 64)
    max_batch_size = min(512, center_batch_size + 64)
    batch_size = round_to_multiple(random_generator.uniform(min_batch_size, max_batch_size), multiple=8)

    center_patience = center_config["patience"]
    min_patience = max(3, center_patience - 2)
    max_patience = min(10, center_patience + 2)
    patience = int(random_generator.integers(min_patience, max_patience + 1))

    center_dropout_rate = center_config["dropout_rate"]

    if center_dropout_rate == 0.0:
        if random_generator.random() < 0.35:
            dropout_rate = 0.0
        else:
            dropout_rate = float(random_generator.uniform(0.01, 0.15))
    else:
        dropout_rate = float(random_generator.uniform(max(0.0, center_dropout_rate - 0.10), min(0.50, center_dropout_rate + 0.10)))


    if random_generator.random() < 0.80:
        activation_name = center_config["activation_name"]
    else:
        activation_name = str(random_generator.choice(ACTIVATION_NAMES))


    if random_generator.random() < 0.80:
        use_batch_normalization = center_config["use_batch_normalization"]
    else:
        use_batch_normalization = bool(random_generator.integers(0, 2))
    

    return {
        "hidden_sizes": hidden_sizes,
        "activation_name": activation_name,
        "use_batch_normalization":
            use_batch_normalization,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "patience": patience
    }

In [33]:
REFINED_CENTER_COUNT = 3
REFINED_TRIALS_PER_CENTER = 6
REFINED_MAXIMUM_EPOCHS = 60

broad_coarse_results = torch.load(
    "broad_coarse_results.pt",
    weights_only=False
)

baseline_refined_center = {
    "trial": 0,
    "config": {
        "hidden_sizes":
            baseline_config["hidden_sizes"].copy(),
        "activation_name":
            baseline_config["activation_name"],
        "use_batch_normalization":
            baseline_config[
                "use_batch_normalization"
            ],
        "dropout_rate":
            baseline_config["dropout_rate"],
        "batch_size":
            BASELINE_BATCH_SIZE,
        "learning_rate":
            BASELINE_LEARNING_RATE,
        "patience": 8
    },
    "best_epoch":
        baseline_training_result["best_epoch"],
    "validation_loss":
        baseline_validation_loss,
    "validation_accuracy":
        baseline_validation_accuracy
}

refined_center_candidates = (
    broad_coarse_results.copy()
)

refined_center_candidates.append(
    baseline_refined_center
)

refined_center_results = sorted(
    refined_center_candidates,
    key=lambda result: (
        result["validation_loss"],
        -result["validation_accuracy"]
    )
)[:REFINED_CENTER_COUNT]

refined_random_generator = np.random.default_rng(SEED + 200)
refined_results = []

refined_trial_index = 1
total_refined_trials = (REFINED_CENTER_COUNT * REFINED_TRIALS_PER_CENTER)

for center_rank, center_result in enumerate(refined_center_results, start=1):
    center_config = center_result["config"]

    for local_trial_index in range(1, REFINED_TRIALS_PER_CENTER + 1):

        refined_config = generate_refined_config(center_config, refined_random_generator)
    
        print(
            f"\nRefined trial "
            f"{refined_trial_index} / {total_refined_trials}"
        )
        print("Center rank:", center_rank)
        print(refined_config)

        experiment_result = run_experiment(
            config=refined_config,
            experiment_name="refined_temporary",
            maximum_epochs=REFINED_MAXIMUM_EPOCHS,
            seed=SEED,
            verbose=False
        )

        refined_results.append({
            "trial": refined_trial_index,
            "center_rank": center_rank,
            "center_config": center_config.copy(),
            "config": refined_config.copy(),
            "best_epoch": experiment_result["best_epoch"],
            "validation_loss": experiment_result["validation_loss"],
            "validation_accuracy": experiment_result["validation_accuracy"]
        })

        torch.save(refined_results, "refined_results.pt")

        print(
            f"Best epoch: "
            f"{experiment_result['best_epoch']} | "
            f"Validation loss: "
            f"{experiment_result['validation_loss']:.4f} | "
            f"Validation accuracy: "
            f"{experiment_result['validation_accuracy']:.4f}"
        )

        refined_trial_index += 1

        del experiment_result

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    


print(
    "\nCompleted refined trials:",
    len(refined_results)
)


Refined trial 1 / 18
Center rank: 1
{'hidden_sizes': [976, 608, 416, 336, 192], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.24268785891783548, 'batch_size': 80, 'learning_rate': 0.0008844402542071969, 'patience': 3}
Best epoch: 10 | Validation loss: 0.0636 | Validation accuracy: 0.9824

Refined trial 2 / 18
Center rank: 1
{'hidden_sizes': [912, 672, 576, 256], 'activation_name': 'leaky_relu', 'use_batch_normalization': False, 'dropout_rate': 0.2844078218072577, 'batch_size': 32, 'learning_rate': 0.001387069844137857, 'patience': 5}
Best epoch: 7 | Validation loss: 0.1102 | Validation accuracy: 0.9719

Refined trial 3 / 18
Center rank: 1
{'hidden_sizes': [960, 704, 576, 320], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.1069356991691604, 'batch_size': 64, 'learning_rate': 0.0005728012865720052, 'patience': 6}
Best epoch: 12 | Validation loss: 0.0633 | Validation accuracy: 0.9841

Refined trial 4 / 18
Center rank: 1
{'hi

In [34]:
broad_coarse_results = torch.load(
    "broad_coarse_results.pt",
    weights_only=False
)

refined_results = torch.load(
    "refined_results.pt",
    weights_only=False
)

combined_results = []

combined_results.append({
    "search_stage": "strong_baseline",
    "trial": 0,
    "config": {
        "hidden_sizes":
            baseline_config["hidden_sizes"].copy(),
        "activation_name":
            baseline_config["activation_name"],
        "use_batch_normalization":
            baseline_config[
                "use_batch_normalization"
            ],
        "dropout_rate":
            baseline_config["dropout_rate"],
        "batch_size":
            BASELINE_BATCH_SIZE,
        "learning_rate":
            BASELINE_LEARNING_RATE,
        "patience": 8
    },
    "best_epoch":
        baseline_training_result["best_epoch"],
    "validation_loss":
        baseline_validation_loss,
    "validation_accuracy":
        baseline_validation_accuracy
})

for result in broad_coarse_results:
    combined_results.append({
        "search_stage": "broad_coarse",
        "trial": result["trial"],
        "config": result["config"].copy(),
        "best_epoch": result["best_epoch"],
        "validation_loss": result["validation_loss"],
        "validation_accuracy": result["validation_accuracy"]
    })

for result in refined_results:
    combined_results.append({
        "search_stage": "refined",
        "trial": result["trial"],
        "config": result["config"].copy(),
        "best_epoch": result["best_epoch"],
        "validation_loss": result["validation_loss"],
        "validation_accuracy": result["validation_accuracy"]
    })

combined_results = sorted(
    combined_results,
    key=lambda result: (
    result["validation_loss"],
    -result["validation_accuracy"]
    )
)

torch.save(combined_results, "combined_search_results.pt")

print("Total evaluated configurations:", len(combined_results))
print("\nTop 10 configurations:\n")

for rank, result in enumerate(combined_results[:10], start=1):
    print(f"Rank {rank}")
    print("Search stage:", result["search_stage"])
    print("Trial:", result["trial"])
    print("Configuration:", result["config"])
    print("Best epoch:", result["best_epoch"])
    print(
        "Validation loss:",
        round(result["validation_loss"], 6)
    )
    print(
        "Validation accuracy:",
        round(result["validation_accuracy"], 6)
    )
    print()    

Total evaluated configurations: 59

Top 10 configurations:

Rank 1
Search stage: broad_coarse
Trial: 26
Configuration: {'hidden_sizes': [864, 624, 496, 304], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.20185521672972673, 'batch_size': 40, 'learning_rate': 0.0010606140772019422, 'patience': 4}
Best epoch: 12
Validation loss: 0.058247
Validation accuracy: 0.9851

Rank 2
Search stage: strong_baseline
Trial: 0
Configuration: {'hidden_sizes': [512, 256, 128], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.2, 'batch_size': 128, 'learning_rate': 0.001, 'patience': 8}
Best epoch: 14
Validation loss: 0.061888
Validation accuracy: 0.9837

Rank 3
Search stage: refined
Trial: 9
Configuration: {'hidden_sizes': [496, 288, 112, 96], 'activation_name': 'leaky_relu', 'use_batch_normalization': True, 'dropout_rate': 0.14919840813827812, 'batch_size': 72, 'learning_rate': 0.000881906935782367, 'patience': 9}
Best epoch: 12
Validation loss: 

In [35]:
SEED_VALIDATION_CANDIDATE_COUNT = 3
SEED_VALIDATION_SEEDS = [SEED, SEED + 1, SEED + 2]
SEED_VALIDATION_MAX_EPOCHS = 60

def create_config_signature(config):
    return (
        tuple(config["hidden_sizes"]),
        config["activation_name"],
        config["use_batch_normalization"],
        round(config["dropout_rate"], 12),
        config["batch_size"],
        round(config["learning_rate"], 15),
        config["patience"]
    )

combined_results = torch.load(
    "combined_search_results.pt",
    weights_only=False
)

combined_results = sorted(
    combined_results,
    key=lambda result: (
        result["validation_loss"],
        -result["validation_accuracy"]
    )
)

top_candidate_results = []
seen_configurations = set()

for result in combined_results:
    config_signature = create_config_signature(result["config"])

    if config_signature not in seen_configurations:
        seen_configurations.add(config_signature)
        top_candidate_results.append(result)

    if len(top_candidate_results) == SEED_VALIDATION_CANDIDATE_COUNT:
        break

print("Selected configurations:\n")

for candidate_rank, result in enumerate(top_candidate_results, start=1):
    print(f"Candidate {candidate_rank}")
    print("Search stage:", result["search_stage"])
    print("Trial:", result["trial"])
    print("Configuration:", result["config"])
    print()

seed_validation_results = []

total_runs = (SEED_VALIDATION_CANDIDATE_COUNT * len(SEED_VALIDATION_SEEDS))

run_index = 1

for candidate_rank, candidate_result in enumerate(top_candidate_results, start=1):
    candidate_config = candidate_result["config"].copy()

    for current_seed in SEED_VALIDATION_SEEDS:

        print(
            f"\nSeed validation run "
            f"{run_index} / {total_runs}"
        )
        print("Candidate:", candidate_rank)
        print("Seed:", current_seed)

        experiment_result = run_experiment(
            config=candidate_config,
            experiment_name=(
                f"seed_candidate_{candidate_rank}"
                f"_seed_{current_seed}"
            ),
            maximum_epochs=SEED_VALIDATION_MAX_EPOCHS,
            seed=current_seed,
            verbose=False
        )

        seed_validation_results.append({
            "candidate_rank": candidate_rank,
            "seed": current_seed,
            "config": candidate_config.copy(),
            "best_epoch": experiment_result["best_epoch"],
            "validation_loss":
                experiment_result["validation_loss"],
            "validation_accuracy":
                experiment_result["validation_accuracy"]
        })

        torch.save(seed_validation_results, "seed_validation_results.pt")

        print(
            f"Best epoch: "
            f"{experiment_result['best_epoch']} | "
            f"Validation loss: "
            f"{experiment_result['validation_loss']:.4f} | "
            f"Validation accuracy: "
            f"{experiment_result['validation_accuracy']:.4f}"
        )

        run_index += 1

        del experiment_result

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

seed_validation_summary = []

for candidate_rank in range(1, SEED_VALIDATION_CANDIDATE_COUNT + 1):
    candidate_runs = [
        result
        for result in seed_validation_results
        if result["candidate_rank"] == candidate_rank
    ]

    validation_losses = np.array([
        result["validation_loss"]
        for result in candidate_runs
    ])

    validation_accuracies = np.array([
        result["validation_accuracy"]
        for result in candidate_runs
    ])

    best_epochs = np.array([
        result["best_epoch"]
        for result in candidate_runs
    ])

    seed_validation_summary.append({
        "candidate_rank": candidate_rank,
        "config": candidate_runs[0]["config"].copy(),
        "mean_validation_loss":
            float(validation_losses.mean()),
        "std_validation_loss":
            float(validation_losses.std()),
        "mean_validation_accuracy":
            float(validation_accuracies.mean()),
        "std_validation_accuracy":
            float(validation_accuracies.std()),
        "mean_best_epoch":
            float(best_epochs.mean())
    })


seed_validation_summary = sorted(
    seed_validation_summary,
    key=lambda result: (
        result["mean_validation_loss"],
        -result["mean_validation_accuracy"],
        result["std_validation_loss"]
    )
)

torch.save(seed_validation_summary, "seed_validation_summary.pt")
print("\nSeed validation summary:\n")

for rank, result in enumerate(seed_validation_summary, start=1):

    print(f"Rank {rank}")
    print("Configuration:", result["config"])
    print(
        "Mean validation loss:",
        round(result["mean_validation_loss"], 6)
    )
    print(
        "Validation loss standard deviation:",
        round(result["std_validation_loss"], 6)
    )
    print(
        "Mean validation accuracy:",
        round(result["mean_validation_accuracy"], 6)
    )
    print(
        "Validation accuracy standard deviation:",
        round(result["std_validation_accuracy"], 6)
    )
    print(
        "Mean best epoch:",
        round(result["mean_best_epoch"], 2)
    )
    print()

Selected configurations:

Candidate 1
Search stage: broad_coarse
Trial: 26
Configuration: {'hidden_sizes': [864, 624, 496, 304], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.20185521672972673, 'batch_size': 40, 'learning_rate': 0.0010606140772019422, 'patience': 4}

Candidate 2
Search stage: strong_baseline
Trial: 0
Configuration: {'hidden_sizes': [512, 256, 128], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.2, 'batch_size': 128, 'learning_rate': 0.001, 'patience': 8}

Candidate 3
Search stage: refined
Trial: 9
Configuration: {'hidden_sizes': [496, 288, 112, 96], 'activation_name': 'leaky_relu', 'use_batch_normalization': True, 'dropout_rate': 0.14919840813827812, 'batch_size': 72, 'learning_rate': 0.000881906935782367, 'patience': 9}


Seed validation run 1 / 9
Candidate: 1
Seed: 42
Best epoch: 12 | Validation loss: 0.0582 | Validation accuracy: 0.9851

Seed validation run 2 / 9
Candidate: 1
Seed: 43
Best epoch: 14 | Va

In [36]:
seed_validation_summary = torch.load("seed_validation_summary.pt", weights_only=False)
best_seed_validation_result = seed_validation_summary[0]
final_config = best_seed_validation_result["config"].copy()

print("Selected final configuration: ")
print(final_config)
print("Mean validation loss: ", best_seed_validation_result["mean_validation_loss"])
print("Mean validation accuracy: ", best_seed_validation_result["mean_validation_accuracy"])
print("Mean best epoch: ", best_seed_validation_result["mean_best_epoch"])

final_experiment_result = run_experiment(
    config=final_config,
    experiment_name="final_mlp",
    maximum_epochs=60,
    seed=SEED,
    verbose=True
)

final_model = final_experiment_result["model"]
final_validation_loss = final_experiment_result["validation_loss"]
final_validation_accuracy = final_experiment_result["validation_accuracy"]

print("\nFinal model training completed.")
print("Best epoch: ", final_experiment_result["best_epoch"])
print("Validation loss: ", round(final_validation_loss, 6))
print("Validation accuracy:", round(final_validation_accuracy, 6))
print("Model state saved to: ", final_experiment_result["checkpoint_path"])

Selected final configuration: 
{'hidden_sizes': [864, 624, 496, 304], 'activation_name': 'relu', 'use_batch_normalization': True, 'dropout_rate': 0.20185521672972673, 'batch_size': 40, 'learning_rate': 0.0010606140772019422, 'patience': 4}
Mean validation loss:  0.05925364047288895
Mean validation accuracy:  0.9840999643007914
Mean best epoch:  12.0
Epoch 01 | Train loss: 0.2737 | Train acc: 0.9185 | Validation loss: 0.1273 | Validation acc: 0.9608
Epoch 02 | Train loss: 0.1518 | Train acc: 0.9540 | Validation loss: 0.0892 | Validation acc: 0.9743
Epoch 03 | Train loss: 0.1223 | Train acc: 0.9623 | Validation loss: 0.0910 | Validation acc: 0.9724
Epoch 04 | Train loss: 0.0993 | Train acc: 0.9697 | Validation loss: 0.0763 | Validation acc: 0.9779
Epoch 05 | Train loss: 0.0853 | Train acc: 0.9738 | Validation loss: 0.0794 | Validation acc: 0.9774
Epoch 06 | Train loss: 0.0750 | Train acc: 0.9764 | Validation loss: 0.0750 | Validation acc: 0.9777
Epoch 07 | Train loss: 0.0686 | Train acc:

In [37]:
_,_, final_test_loader = create_data_loaders(
    training_batch_size=final_config["batch_size"],
    evaluation_batch_size=1024,
    seed=SEED
)

final_test_loss, final_test_accuracy = evaluate_model(
    model=final_model,
    data_loader=final_test_loader,
    criterion=criterion,
    device=device
)

print("Final test results:")
print("Test loss:", round(final_test_loss, 6))
print("Test accuracy:", round(final_test_accuracy, 6))
print("Test accuracy percentage:", f"{final_test_accuracy * 100:.2f}%")

Final test results:
Test loss: 0.056269
Test accuracy: 0.9834
Test accuracy percentage: 98.34%


In [38]:
baseline_test_loss, baseline_test_accuracy = (
    evaluate_model(
        model=best_baseline_model,
        data_loader=final_test_loader,
        criterion=criterion,
        device=device
    )
)

print("\nStrong baseline test results:")
print("Baseline test loss:", round(baseline_test_loss, 6))
print("Baseline test accuracy:", round(baseline_test_accuracy, 6))
print("Baseline test accuracy percentage:", f"{baseline_test_accuracy * 100:.2f}%")


Strong baseline test results:
Baseline test loss: 0.056573
Baseline test accuracy: 0.9833
Baseline test accuracy percentage: 98.33%


In [39]:
final_model.eval()

true_labels = []
predicted_labels = []

with torch.no_grad():
    for feature_batch, label_batch in final_test_loader:
        feature_batch = feature_batch.to(device, non_blocking=True)
        output_batch = final_model(feature_batch)
        prediction_batch = output_batch.argmax(dim=1)
        true_labels.extend(label_batch.cpu().numpy().tolist())
        predicted_labels.extend(prediction_batch.cpu().numpy().tolist())

true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)

final_confusion_matrix = confusion_matrix(true_labels, predicted_labels)

print("Final MLP confusion matrix: ")
print(final_confusion_matrix)
print("\nCorrect predictions:", int(np.trace(final_confusion_matrix)))
print("Total test samples:", int(final_confusion_matrix.sum()))

Final MLP confusion matrix: 
[[ 971    1    0    0    1    1    2    2    1    1]
 [   0 1125    0    2    0    0    2    3    3    0]
 [   2    0 1017    2    3    0    0    6    2    0]
 [   1    0    4  994    0    0    0    4    3    4]
 [   0    0    2    0  954    0    4    1    1   20]
 [   3    0    0    7    1  873    2    1    4    1]
 [   4    2    1    0    3    4  941    0    3    0]
 [   2    2    7    0    1    0    0 1007    1    8]
 [   1    1    2    3    1    1    0    4  957    4]
 [   2    2    0    3    2    3    0    2    0  995]]

Correct predictions: 9834
Total test samples: 10000


In [40]:
class_sample_counts = final_confusion_matrix.sum(axis=1)
class_correct_counts = np.diag(final_confusion_matrix)
class_accuracies = (class_correct_counts / class_sample_counts)
print("Per-class test results:\n")

for digit in range(10):
    print(
        f"Digit {digit} | "
        f"Correct: {class_correct_counts[digit]} / "
        f"{class_sample_counts[digit]} | "
        f"Accuracy: {class_accuracies[digit]:.4f} "
        f"({class_accuracies[digit] * 100:.2f}%)"
    )

Per-class test results:

Digit 0 | Correct: 971 / 980 | Accuracy: 0.9908 (99.08%)
Digit 1 | Correct: 1125 / 1135 | Accuracy: 0.9912 (99.12%)
Digit 2 | Correct: 1017 / 1032 | Accuracy: 0.9855 (98.55%)
Digit 3 | Correct: 994 / 1010 | Accuracy: 0.9842 (98.42%)
Digit 4 | Correct: 954 / 982 | Accuracy: 0.9715 (97.15%)
Digit 5 | Correct: 873 / 892 | Accuracy: 0.9787 (97.87%)
Digit 6 | Correct: 941 / 958 | Accuracy: 0.9823 (98.23%)
Digit 7 | Correct: 1007 / 1028 | Accuracy: 0.9796 (97.96%)
Digit 8 | Correct: 957 / 974 | Accuracy: 0.9825 (98.25%)
Digit 9 | Correct: 995 / 1009 | Accuracy: 0.9861 (98.61%)


In [41]:
final_model_package = {
    "model_state_dict":
        final_model.state_dict(),
    "input_size": 784,
    "output_size": 10,
    "config":
        final_config.copy(),
    "class_weights":
        class_weights.copy(),
    "history":
        final_experiment_result["history"],
    "best_epoch":
        final_experiment_result["best_epoch"],
    "validation_loss":
        final_validation_loss,
    "validation_accuracy":
        final_validation_accuracy,
    "test_loss":
        final_test_loss,
    "test_accuracy":
        final_test_accuracy,
    "confusion_matrix":
        final_confusion_matrix
}

torch.save(
    final_model_package,
    "final_mlp_complete.pt"
)

print("Complete final model package saved.")
print("File: final_mlp_complete.pt")

Complete final model package saved.
File: final_mlp_complete.pt


In [42]:
model_comparison = [
    {
        "model_name": "KNN + PCA",
        "validation_accuracy": 0.9784,
        "test_accuracy": 0.9752
    },
    {
        "model_name": "Decision Tree + Raw",
        "validation_accuracy": 0.8739,
        "test_accuracy": 0.8857
    },
    {
        "model_name": "Decision Tree + PCA",
        "validation_accuracy": 0.8512,
        "test_accuracy": 0.8603
    },
    {
        "model_name": "Random Forest + Raw",
        "validation_accuracy": 0.9697,
        "test_accuracy": 0.9716
    },
    {
        "model_name": "Random Forest + PCA",
        "validation_accuracy": 0.9536,
        "test_accuracy": 0.9575
    },
    {
        "model_name": "Gradient Boosting + Raw",
        "validation_accuracy": 0.9756,
        "test_accuracy": 0.9755
    },
    {
        "model_name": "Gradient Boosting + PCA",
        "validation_accuracy": 0.9576,
        "test_accuracy": 0.9416
    },
    {
    "model_name": "Strong MLP Baseline",
    "validation_accuracy":
        baseline_validation_accuracy,
    "test_accuracy":
        baseline_test_accuracy
    },
    {
        "model_name": "Final MLP + Raw",
        "validation_accuracy":
            final_validation_accuracy,
        "test_accuracy":
            final_test_accuracy
    }
]

model_comparison = sorted(
    model_comparison,
    key=lambda result: result["test_accuracy"],
    reverse=True
)

print(
    f"{'Rank':<6}"
    f"{'Model':<30}"
    f"{'Validation':<15}"
    f"{'Test':<15}"
)

print("-" * 66)

for rank, result in enumerate(
    model_comparison,
    start=1
):
    print(
        f"{rank:<6}"
        f"{result['model_name']:<30}"
        f"{result['validation_accuracy']:<15.4f}"
        f"{result['test_accuracy']:<15.4f}"
    )

Rank  Model                         Validation     Test           
------------------------------------------------------------------
1     Final MLP + Raw               0.9851         0.9834         
2     Strong MLP Baseline           0.9837         0.9833         
3     Gradient Boosting + Raw       0.9756         0.9755         
4     KNN + PCA                     0.9784         0.9752         
5     Random Forest + Raw           0.9697         0.9716         
6     Random Forest + PCA           0.9536         0.9575         
7     Gradient Boosting + PCA       0.9576         0.9416         
8     Decision Tree + Raw           0.8739         0.8857         
9     Decision Tree + PCA           0.8512         0.8603         


In [43]:
previous_model_results = [
    result
    for result in model_comparison
    if result["model_name"] not in {
        "Final MLP + Raw",
        "Strong MLP Baseline"
    }
]

best_previous_model = max(
    previous_model_results,
    key=lambda result:
    result["test_accuracy"]
)

accuracy_difference = (final_test_accuracy - best_previous_model["test_accuracy"])

print("Best previous model:", best_previous_model["model_name"])
print("Best previous test accuracy:", round(best_previous_model["test_accuracy"], 6))
print("Final MLP test accuracy:", round(final_test_accuracy, 6))
print("Accuracy difference:", round(accuracy_difference, 6))
print("Percentage-point difference:", f"{accuracy_difference * 100:+.2f}")

baseline_accuracy_difference = (
    final_test_accuracy
    - baseline_test_accuracy
)

print(
    "\nStrong baseline test accuracy:",
    round(baseline_test_accuracy, 6)
)

print(
    "Tuned MLP improvement over baseline:",
    f"{baseline_accuracy_difference * 100:+.2f} "
    "percentage points"
)

Best previous model: Gradient Boosting + Raw
Best previous test accuracy: 0.9755
Final MLP test accuracy: 0.9834
Accuracy difference: 0.0079
Percentage-point difference: +0.79

Strong baseline test accuracy: 0.9833
Tuned MLP improvement over baseline: +0.01 percentage points


In [44]:
saved_broad_results = torch.load("broad_coarse_results.pt", weights_only=False)
saved_refined_results = torch.load("refined_results.pt", weights_only=False)
saved_combined_results = torch.load("combined_search_results.pt", weights_only=False)
saved_seed_results = torch.load("seed_validation_results.pt", weights_only=False)

assert len(saved_broad_results) == 40
assert len(saved_refined_results) == 18
assert len(saved_combined_results) == 59
assert len(saved_seed_results) == 9
assert final_confusion_matrix.shape == (10,10)
assert int(final_confusion_matrix.sum()) == 10000

confusion_matrix_accuracy = (
    np.trace(final_confusion_matrix)
    / final_confusion_matrix.sum()
)

assert np.isclose(
    confusion_matrix_accuracy,
    final_test_accuracy
)

print("All final checks passed.")

All final checks passed.


##############################################

In [92]:
model_train_images = model_train_vectors.reshape(-1, 28, 28)
model_train_images = model_train_images.astype(np.float32)

augmentation_train_labels = model_train_labels.astype(np.int64)

print("Train images: ", model_train_images.shape)
print("Train labels: ", augmentation_train_labels.shape)
print("Pixel range: ", model_train_images.min(), model_train_images.max())

Train images:  (50000, 28, 28)
Train labels:  (50000,)
Pixel range:  0.0 1.0


In [93]:
def create_patch_mosaic(class_images, random_generator):
    selected_indices = random_generator.choice(len(class_images), size=4, replace=False)
    selected_images = class_images[selected_indices]
    mosaic_image = np.zeros((28,28), dtype=np.float32)

    mosaic_image[:14, :14] = selected_images[0, :14, :14]
    mosaic_image[:14, 14:] = selected_images[1, :14, 14:]
    mosaic_image[14:, :14] = selected_images[2, 14:, :14]
    mosaic_image[14:, 14:] = selected_images[3, 14:, 14:]

    return mosaic_image

In [94]:
def generate_patch_mosaic_data(images, labels, synthetic_per_class, seed=SEED):
    random_generator = np.random.default_rng(seed)

    synthetic_images = []
    synthetic_labels = []

    for class_label in range(10):
        class_images = images[labels == class_label]

        for _ in range(synthetic_per_class):
            synthetic_images.append(create_patch_mosaic(class_images, random_generator))
            synthetic_labels.append(class_label)
    return (np.array(synthetic_images, dtype=np.float32), np.array(synthetic_labels, dtype=np.int64))
        

In [95]:
PATCH_MOSAIC_PER_CLASS_VALUES = [100, 250, 500, 1000]

augmentation_experiments = []

for synthetic_per_class in PATCH_MOSAIC_PER_CLASS_VALUES:
    augmentation_experiments.append({
        "name": f"patch_mosaic_{synthetic_per_class}",
        "method": "patch_mosaic",
        "synthetic_per_class": synthetic_per_class
    })

print("Experiment count: ", len(augmentation_experiments))

for experiment in augmentation_experiments:
    print(experiment)

Experiment count:  4
{'name': 'patch_mosaic_100', 'method': 'patch_mosaic', 'synthetic_per_class': 100}
{'name': 'patch_mosaic_250', 'method': 'patch_mosaic', 'synthetic_per_class': 250}
{'name': 'patch_mosaic_500', 'method': 'patch_mosaic', 'synthetic_per_class': 500}
{'name': 'patch_mosaic_1000', 'method': 'patch_mosaic', 'synthetic_per_class': 1000}


In [96]:
augmentation_generators = {"patch_mosaic": generate_patch_mosaic_data}

In [97]:
def prepare_augmentation_experiment(experiment, seed=SEED):
    generator_func = augmentation_generators[
        experiment["method"]
    ]

    synthetic_images, synthetic_labels = generator_func(
        images=model_train_images,
        labels=augmentation_train_labels,
        synthetic_per_class=experiment["synthetic_per_class"],
        seed=seed
    )

    include_original = experiment.get(
        "include_original",
        True
    )

    if include_original:
        training_images = np.concatenate(
            [model_train_images, synthetic_images],
            axis=0
        )

        training_labels = np.concatenate(
            [augmentation_train_labels, synthetic_labels],
            axis=0
        )
    else:
        training_images = synthetic_images
        training_labels = synthetic_labels

    training_vectors = training_images.reshape(-1, 784)

    training_dataset = TensorDataset(
        torch.from_numpy(
            training_vectors.astype(np.float32)
        ),
        torch.from_numpy(
            training_labels.astype(np.int64)
        )
    )

    class_counts = np.bincount(
        training_labels,
        minlength=10
    )

    class_weights = (
        len(training_labels) / (10 * class_counts)
    )

    experiment_criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=device
        )
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    training_loader = DataLoader(
        training_dataset,
        batch_size=final_config["batch_size"],
        shuffle=True,
        generator=loader_generator,
        pin_memory=device.type == "cuda"
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=1024,
        shuffle=False,
        pin_memory=device.type == "cuda"
    )

    return {
        "training_loader": training_loader,
        "validation_loader": validation_loader,
        "criterion": experiment_criterion,
        "synthetic_count": len(synthetic_labels),
        "original_count": (
            len(model_train_images)
            if include_original
            else 0
        ),
        "total_training_count": len(training_labels)
    }

In [98]:
def run_augmentation_experiment(experiment, max_epoch=60, seed=SEED, verbose=False):
    set_experiment_seed(seed)

    prepared_data = prepare_augmentation_experiment(experiment=experiment, seed=seed)

    model = MLP(
        input_size=784,
        hidden_sizes=final_config["hidden_sizes"],
        output_size=10,
        activation_name=final_config["activation_name"],
        use_batch_normalization=final_config["use_batch_normalization"],
        dropout_rate=final_config["dropout_rate"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=final_config["learning_rate"])
    checkpoint_path = (f"{experiment['name']}_best_model.pt")

    training_result = train_with_early_stopping(
        model=model,
        training_loader=prepared_data["training_loader"],
        validation_loader=prepared_data["validation_loader"],
        criterion=prepared_data["criterion"],
        optimizer=optimizer,
        device=device,
        maximum_epochs=max_epoch,
        patience=final_config["patience"],
        checkpoint_path=checkpoint_path,
        verbose=verbose
    )

    validation_loss, validation_accuracy = evaluate_model(
        model=training_result["model"],
        data_loader=prepared_data["validation_loader"],
        criterion=prepared_data["criterion"],
        device=device
    )

    return {
        "experiment": experiment.copy(),
        "best_epoch": training_result["best_epoch"],
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "synthetic_count": prepared_data["synthetic_count"],
        "total_training_count":
            prepared_data["total_training_count"],
        "checkpoint_path": checkpoint_path
    }

In [99]:
patch_mosaic_amount_results = []

for experiment_index, experiment in enumerate(augmentation_experiments, start=1):

    print(f"\nExperiment {experiment_index} / "f"{len(augmentation_experiments)}")
    print(experiment)

    result = run_augmentation_experiment(
        experiment=experiment,
        max_epoch=60,
        seed=SEED,
        verbose=False
    )

    patch_mosaic_amount_results.append(result)

    torch.save(patch_mosaic_amount_results, "patch_mosaic_amount_results.pt")

    print("Synthetic samples:", result["synthetic_count"])
    print("Total training samples:", result["total_training_count"])
    print("Best epoch:", result["best_epoch"])
    print("Validation loss:", round(result["validation_loss"], 6))
    print("Validation accuracy:", f"{result['validation_accuracy'] * 100:.2f}%")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll amount experiments completed.") 


Experiment 1 / 4
{'name': 'patch_mosaic_100', 'method': 'patch_mosaic', 'synthetic_per_class': 100}
Synthetic samples: 1000
Total training samples: 51000
Best epoch: 7
Validation loss: 0.064834
Validation accuracy: 98.04%

Experiment 2 / 4
{'name': 'patch_mosaic_250', 'method': 'patch_mosaic', 'synthetic_per_class': 250}
Synthetic samples: 2500
Total training samples: 52500
Best epoch: 7
Validation loss: 0.064522
Validation accuracy: 98.15%

Experiment 3 / 4
{'name': 'patch_mosaic_500', 'method': 'patch_mosaic', 'synthetic_per_class': 500}
Synthetic samples: 5000
Total training samples: 55000
Best epoch: 8
Validation loss: 0.060197
Validation accuracy: 98.15%

Experiment 4 / 4
{'name': 'patch_mosaic_1000', 'method': 'patch_mosaic', 'synthetic_per_class': 1000}
Synthetic samples: 10000
Total training samples: 60000
Best epoch: 10
Validation loss: 0.060311
Validation accuracy: 98.20%

All amount experiments completed.


In [100]:
patch_mosaic_amount_results = torch.load(
    "patch_mosaic_amount_results.pt",
    weights_only=False
)

patch_mosaic_amount_results = sorted(
    patch_mosaic_amount_results,
    key=lambda result: (
        result["validation_loss"],
        -result["validation_accuracy"]
    )
)

print(
    "Raw MLP validation accuracy:",
    f"{final_validation_accuracy * 100:.2f}%"
)

print("\nPatch-mosaic amount ranking:\n")

for rank, result in enumerate(
    patch_mosaic_amount_results,
    start=1
):
    amount = result["experiment"]["synthetic_per_class"]

    print(f"Rank {rank}")
    print("Synthetic per class:", amount)
    print("Total synthetic:", result["synthetic_count"])
    print("Best epoch:", result["best_epoch"])
    print(
        "Validation loss:",
        round(result["validation_loss"], 6)
    )
    print(
        "Validation accuracy:",
        f"{result['validation_accuracy'] * 100:.2f}%"
    )
    print()

best_patch_mosaic_amount_result = (
    patch_mosaic_amount_results[0]
)

BEST_PATCH_MOSAIC_PER_CLASS = (
    best_patch_mosaic_amount_result[
        "experiment"
    ]["synthetic_per_class"]
)

print(
    "Selected synthetic amount per class:",
    BEST_PATCH_MOSAIC_PER_CLASS
)

Raw MLP validation accuracy: 98.51%

Patch-mosaic amount ranking:

Rank 1
Synthetic per class: 500
Total synthetic: 5000
Best epoch: 8
Validation loss: 0.060197
Validation accuracy: 98.15%

Rank 2
Synthetic per class: 1000
Total synthetic: 10000
Best epoch: 10
Validation loss: 0.060311
Validation accuracy: 98.20%

Rank 3
Synthetic per class: 250
Total synthetic: 2500
Best epoch: 7
Validation loss: 0.064522
Validation accuracy: 98.15%

Rank 4
Synthetic per class: 100
Total synthetic: 1000
Best epoch: 7
Validation loss: 0.064834
Validation accuracy: 98.04%

Selected synthetic amount per class: 500


In [101]:
best_patch_experiment = (best_patch_mosaic_amount_result["experiment"].copy())
best_patch_data = prepare_augmentation_experiment(experiment=best_patch_experiment, seed=SEED)
best_patch_mosaic_model = MLP(
    input_size=784,
    hidden_sizes=final_config["hidden_sizes"],
    output_size=10,
    activation_name=final_config["activation_name"],
    use_batch_normalization=final_config["use_batch_normalization"],
    dropout_rate = final_config["dropout_rate"]
).to(device)

best_model_state = torch.load(
    best_patch_mosaic_amount_result["checkpoint_path"],
    weights_only=True,
    map_location=device
)

best_patch_mosaic_model.load_state_dict(best_model_state)

best_patch_loader = DataLoader(
    test_dataset, 
    batch_size=1024, 
    shuffle=False, 
    pin_memory = device.type== "cuda"
)

best_patch_validation_loss, best_patch_validation_accuracy = (
    evaluate_model(
        model=best_patch_mosaic_model,
        data_loader=best_patch_data["validation_loader"],
        criterion=best_patch_data["criterion"],
        device=device
    )
)

best_patch_test_loss, best_patch_test_accuracy = evaluate_model(
    model=best_patch_mosaic_model,
    data_loader=final_test_loader,
    criterion=best_patch_data["criterion"],
    device=device
)


print("Selected synthetic per class:", BEST_PATCH_MOSAIC_PER_CLASS)
print("Total synthetic samples:",best_patch_mosaic_amount_result["synthetic_count"])
print("Best epoch:", best_patch_mosaic_amount_result["best_epoch"])
print("Validation accuracy:", f"{best_patch_validation_accuracy * 100:.2f}%")
print("Test accuracy:", f"{best_patch_test_accuracy * 100:.2f}%")


Selected synthetic per class: 500
Total synthetic samples: 5000
Best epoch: 8
Validation accuracy: 98.15%
Test accuracy: 98.32%


In [102]:
best_patch_true_labels = []
best_patch_predictions = []

best_patch_mosaic_model.eval()

with torch.inference_mode():
    for feature_batch, label_batch in final_test_loader:
        feature_batch = feature_batch.to(device, non_blocking=True)

        outputs = best_patch_mosaic_model(feature_batch)
        predictions = outputs.argmax(dim=1)

        best_patch_true_labels.extend(label_batch.cpu().numpy())
        best_patch_predictions.extend(predictions.cpu().numpy())

best_patch_confusion_matrix = confusion_matrix(best_patch_true_labels, best_patch_predictions)

class_sample_counts = best_patch_confusion_matrix.sum(axis=1)
class_correct_counts = np.diag(best_patch_confusion_matrix)
best_patch_class_accuracies = (class_correct_counts / class_sample_counts)

print("Best patch-mosaic confusion matrix:")
print(best_patch_confusion_matrix)
print("\nPer-class accuracies:")

for digit in range(10):
    print(f"Digit {digit}: "f"{best_patch_class_accuracies[digit] * 100:.2f}%")

patch_accuracy_difference = (best_patch_test_accuracy - final_test_accuracy)

print("\nRaw MLP test accuracy:", f"{final_test_accuracy * 100:.2f}%")
print("Best patch-mosaic test accuracy:", f"{best_patch_test_accuracy * 100:.2f}%")
print("Difference:", f"{patch_accuracy_difference * 100:+.2f} " "percentage points")

Best patch-mosaic confusion matrix:
[[ 972    1    0    0    0    1    3    0    1    2]
 [   0 1127    2    3    0    0    1    1    1    0]
 [   2    0 1021    1    1    0    0    4    3    0]
 [   0    0    4  990    0    3    0    6    2    5]
 [   1    0    4    0  963    0    3    2    0    9]
 [   3    0    0    8    0  875    4    1    1    0]
 [   3    4    0    1    3    3  941    0    3    0]
 [   0    3    7    1    0    0    0 1003    5    9]
 [   5    0    3    2    3    2    0    5  954    0]
 [   4    2    1    3    6    2    0    2    3  986]]

Per-class accuracies:
Digit 0: 99.18%
Digit 1: 99.30%
Digit 2: 98.93%
Digit 3: 98.02%
Digit 4: 98.07%
Digit 5: 98.09%
Digit 6: 98.23%
Digit 7: 97.57%
Digit 8: 97.95%
Digit 9: 97.72%

Raw MLP test accuracy: 98.34%
Best patch-mosaic test accuracy: 98.32%
Difference: -0.02 percentage points


In [103]:
best_patch_mosaic_package = {
    "model_state_dict":
        best_patch_mosaic_model.state_dict(),
    "model_config":
        final_config.copy(),
    "augmentation_method":
        "14x14 patch mosaic",
    "synthetic_per_class":
        BEST_PATCH_MOSAIC_PER_CLASS,
    "total_synthetic_samples":
        best_patch_mosaic_amount_result["synthetic_count"],
    "total_training_samples":
        best_patch_mosaic_amount_result[
            "total_training_count"
        ],
    "best_epoch":
        best_patch_mosaic_amount_result["best_epoch"],
    "validation_loss":
        best_patch_validation_loss,
    "validation_accuracy":
        best_patch_validation_accuracy,
    "test_loss":
        best_patch_test_loss,
    "test_accuracy":
        best_patch_test_accuracy,
    "confusion_matrix":
        best_patch_confusion_matrix,
    "amount_search_results":
        patch_mosaic_amount_results
}

torch.save(
    best_patch_mosaic_package,
    "best_patch_mosaic_mlp_complete.pt"
)

patch_model_name = (
    "MLP + Best 14x14 Patch Mosaic"
)

model_comparison = [
    result
    for result in model_comparison
    if result["model_name"] != patch_model_name
]

model_comparison.append({
    "model_name":
        patch_model_name,
    "validation_accuracy":
        best_patch_validation_accuracy,
    "test_accuracy":
        best_patch_test_accuracy
})

print("Patch-mosaic research completed.")
print("Selected amount:", BEST_PATCH_MOSAIC_PER_CLASS)
print("Saved file: best_patch_mosaic_mlp_complete.pt")

Patch-mosaic research completed.
Selected amount: 500
Saved file: best_patch_mosaic_mlp_complete.pt


In [104]:
SYNTHETIC_ONLY_PER_CLASS_VALUES = sorted(set([BEST_PATCH_MOSAIC_PER_CLASS, len(model_train_images) // 10]))
synthetic_only_experiments = []

for synthetic_per_class in SYNTHETIC_ONLY_PER_CLASS_VALUES:
    synthetic_only_experiments.append({
        "name": f"patch_mosaic_only_{synthetic_per_class}",
        "method": "patch_mosaic",
        "synthetic_per_class": synthetic_per_class,
        "include_original": False
    })

print("Synthetic-only experiment count: ", len(synthetic_only_experiments))

for experiment in synthetic_only_experiments:
    print(experiment)

Synthetic-only experiment count:  2
{'name': 'patch_mosaic_only_500', 'method': 'patch_mosaic', 'synthetic_per_class': 500, 'include_original': False}
{'name': 'patch_mosaic_only_5000', 'method': 'patch_mosaic', 'synthetic_per_class': 5000, 'include_original': False}


In [105]:
synthetic_only_results = []

for experiment_index, experiment in enumerate(synthetic_only_experiments, start=1):
    print(f"\nExperiment {experiment_index} / "f"{len(synthetic_only_experiments)}")
    print(experiment)

    result = run_augmentation_experiment(
        experiment=experiment,
        max_epoch=60,
        seed=SEED,
        verbose=False
    )

    synthetic_only_model = MLP(
        input_size=784,
        hidden_sizes=final_config["hidden_sizes"],
        output_size=10,
        activation_name=final_config["activation_name"],
        use_batch_normalization=final_config["use_batch_normalization"],
        dropout_rate=final_config["dropout_rate"]
    ).to(device)

    synthetic_only_state = torch.load(result["checkpoint_path"], weights_only=True, map_location=device)
    
    synthetic_only_model.load_state_dict(synthetic_only_state)

    test_loss, test_accuracy = evaluate_model(
        model=synthetic_only_model,
        data_loader=final_test_loader,
        criterion=nn.CrossEntropyLoss(),
        device=device
    ) 

    result["test_loss"] = test_loss
    result["test_accuracy"] = test_accuracy
    synthetic_only_results.append(result)

    print(
        "Synthetic samples:",
        result["synthetic_count"]
    )
    print(
        "Total training samples:",
        result["total_training_count"]
    )
    print(
        "Best epoch:",
        result["best_epoch"]
    )
    print(
        "Validation loss:",
        round(result["validation_loss"], 6)
    )
    print(
        "Validation accuracy:",
        f"{result['validation_accuracy'] * 100:.2f}%"
    )
    print(
        "Real test accuracy:",
        f"{result['test_accuracy'] * 100:.2f}%"
    )
    print(
        "Checkpoint:",
        result["checkpoint_path"]
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

synthetic_only_results = sorted(
    synthetic_only_results, 
    key=lambda result: result["synthetic_count"]
)

torch.save(synthetic_only_results, "patch_mosaic_synthetic_only_results.pt")

print("\nAll synthetic-only experiments completed.")


Experiment 1 / 2
{'name': 'patch_mosaic_only_500', 'method': 'patch_mosaic', 'synthetic_per_class': 500, 'include_original': False}
Synthetic samples: 5000
Total training samples: 5000
Best epoch: 1
Validation loss: 0.412024
Validation accuracy: 87.66%
Real test accuracy: 88.58%
Checkpoint: patch_mosaic_only_500_best_model.pt

Experiment 2 / 2
{'name': 'patch_mosaic_only_5000', 'method': 'patch_mosaic', 'synthetic_per_class': 5000, 'include_original': False}
Synthetic samples: 50000
Total training samples: 50000
Best epoch: 8
Validation loss: 0.247355
Validation accuracy: 94.17%
Real test accuracy: 94.68%
Checkpoint: patch_mosaic_only_5000_best_model.pt

All synthetic-only experiments completed.


In [106]:
print("Training-data comparison:\n")

print("1. Raw MLP")
print("Training composition:",f"{len(model_train_images)} real + 0 synthetic")
print("Validation accuracy:",f"{final_validation_accuracy * 100:.2f}%")
print("Test accuracy:",f"{final_test_accuracy * 100:.2f}%")

print("\n2. Real + best patch-mosaic")
print( "Training composition:",f"{len(model_train_images)} real + "f"{best_patch_mosaic_amount_result['synthetic_count']} synthetic")
print("Validation accuracy:",f"{best_patch_validation_accuracy * 100:.2f}%")
print("Test accuracy:",f"{best_patch_test_accuracy * 100:.2f}%")

for result in synthetic_only_results:
    synthetic_count = result["synthetic_count"]

    print(f"\n3. Synthetic-only — "f"{synthetic_count} samples")
    print("Training composition:",f"0 real + {synthetic_count} synthetic")
    print("Best epoch:",result["best_epoch"])
    print("Validation accuracy:",f"{result['validation_accuracy'] * 100:.2f}%")
    print("Real test accuracy:",f"{result['test_accuracy'] * 100:.2f}%")
    print("Difference from raw MLP:",f"{(result['test_accuracy'] - final_test_accuracy) * 100:+.2f} " "percentage points")

small_result = synthetic_only_results[0]
large_result = synthetic_only_results[1]

synthetic_amount_effect = (
    large_result["test_accuracy"]
    - small_result["test_accuracy"]
)

print(
    "\nEffect of increasing synthetic data "
    "from 5,000 to 50,000:",
    f"{synthetic_amount_effect * 100:+.2f} "
    "percentage points"
)

Training-data comparison:

1. Raw MLP
Training composition: 50000 real + 0 synthetic
Validation accuracy: 98.51%
Test accuracy: 98.34%

2. Real + best patch-mosaic
Training composition: 50000 real + 5000 synthetic
Validation accuracy: 98.15%
Test accuracy: 98.32%

3. Synthetic-only — 5000 samples
Training composition: 0 real + 5000 synthetic
Best epoch: 1
Validation accuracy: 87.66%
Real test accuracy: 88.58%
Difference from raw MLP: -9.76 percentage points

3. Synthetic-only — 50000 samples
Training composition: 0 real + 50000 synthetic
Best epoch: 8
Validation accuracy: 94.17%
Real test accuracy: 94.68%
Difference from raw MLP: -3.66 percentage points

Effect of increasing synthetic data from 5,000 to 50,000: +6.10 percentage points
